# Fractional delayed fault model — Colab runner (results saved in Google Drive)

Runs the whole numerical study (version v2: multi-start optimiser, information-matched baseline, bootstrap coupling test) of the article in parallel and saves **everything in `MyDrive/fault_study/`**
(`out/` raw results, `figs/`, `tabs/`, `results.xlsx`, `summary.json`, `article_bundle.zip`).

* **Alone:** leave `CORES = ''` — Colab uses all its cores.
* **Together with your desktop:** set the same `CORES` on both machines (e.g. `'2,8'` = Colab cores, desktop cores) and `ME = 0` here, `ME = 1` on the desktop (see the README).
* Everything is **resumable**: after a disconnect run the cells again, finished tasks are skipped.

Cells: 1 setup + run · 2 final analysis (run it when `TOTAL n/n` is printed) · 3 optional PDF.

In [ ]:
# 1. SETUP + RUN (parallel, resumable)
MODE  = 'FULL'   # 'QUICK' = few-minute check, 'FULL' = the study of the article
CORES = ''       # '' = this machine only; '2,8' = shared with a desktop (same string on both machines)
ME    = 0        # index of this machine in CORES (Colab = 0, desktop = 1)

import os, subprocess, getpass, multiprocessing
from google.colab import drive
drive.mount('/content/drive')
SHARED = '/content/drive/MyDrive/fault_study'
for d in ('out', 'figs', 'tabs'):
    os.makedirs(f'{SHARED}/{d}', exist_ok=True)

REPO, SRC = 'yamenetoo/fractional-delayed-fault-model', '/content/repo'
if os.path.isdir(f'{SRC}/.git'):
    subprocess.run('git pull -q', shell=True, cwd=SRC)
else:
    r = subprocess.run(f'git clone -q https://github.com/{REPO} {SRC}', shell=True, capture_output=True, text=True)
    if r.returncode:
        tok = getpass.getpass('Repository is private - GitHub token (hidden): ')
        subprocess.run(f'git clone -q https://{tok}@github.com/{REPO} {SRC}', shell=True)
!pip -q install numba openpyxl pymupdf
!apt-get -qq install -y libgeos-dev >/dev/null
!pip -q install cartopy
NCPU = multiprocessing.cpu_count()
print('Colab cores:', NCPU)
CFG = dict(QUICK=dict(R1=3,B1=6,R2=3,B2=6,M=2,G1=6,G2=6), FULL=dict(R1=24,B1=40,R2=16,B2=24,M=12,G1=40,G2=24))[MODE]
SPLIT = f'--cores {CORES} --me {ME} --steal' if CORES else f'--procs {NCPU}'
!cd {SRC}/stats && python -u run_all.py {SPLIT} --R1 {CFG['R1']} --B1 {CFG['B1']} --R2 {CFG['R2']} --B2 {CFG['B2']} --M {CFG['M']} --G1 {CFG['G1']} --G2 {CFG['G2']} --npart 100 --out {SHARED}/out

In [ ]:
# 2. FINAL ANALYSIS: figures, tables, Excel, article bundle -> everything saved in Drive
import os, sys, glob, shutil, zipfile, subprocess
FAULT1, FAULT2 = '', ''          # '' = automatic choice of two neighbouring traces (~30 km apart); or substrings of GEM names
ST = f'{SRC}/stats'
env = dict(os.environ, FAULT1=FAULT1, FAULT2=FAULT2)
def run(*a):
    subprocess.run([sys.executable, *a], cwd=ST, env=env, check=True)

run('run_all.py', '--status', '--out', f'{SHARED}/out')       # progress (all tasks must be finished)
run('run_all.py', '--merge', '--out', f'{SHARED}/out')        # merge robustness parts
shutil.copytree(f'{SHARED}/out', f'{ST}/out', dirs_exist_ok=True)   # local copy = fast and safe
for s in ('fault_map.py', 'fig_basic.py', 'analyze.py', 'export_excel.py'):
    run(s)
for sub in ('figs', 'tabs'):
    shutil.copytree(f'{ST}/{sub}', f'{SHARED}/{sub}', dirs_exist_ok=True)
for f in ('summary.json', 'results.xlsx', 'out_fault_pair.json'):
    if os.path.exists(f'{ST}/{f}'):
        shutil.copy(f'{ST}/{f}', f'{SHARED}/{f}')
with zipfile.ZipFile(f'{SHARED}/article_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{SRC}/paper/*.tex'):
        z.write(f, 'paper/' + os.path.basename(f))
    z.write(f'{ST}/schematic.tex', 'stats/schematic.tex')
    for sub in ('figs', 'tabs'):
        for f in glob.glob(f'{SHARED}/{sub}/*'):
            z.write(f, f'stats/{sub}/' + os.path.basename(f))
import fitz
from IPython.display import Image, display
for f in sorted(glob.glob(f'{SHARED}/figs/*.pdf')):
    png = f[:-4] + '.png'
    fitz.open(f)[0].get_pixmap(dpi=100).save(png)
    print(os.path.basename(f)); display(Image(png))
print('SAVED IN DRIVE:', SHARED)

In [ ]:
# 3. (optional) compile the article PDF in Colab and save it in Drive (~3-5 min for the LaTeX install)
!apt-get -qq install -y texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended >/dev/null
!cd {SRC}/paper && pdflatex -interaction=nonstopmode article.tex >/dev/null; cd {SRC}/paper && pdflatex -interaction=nonstopmode article.tex | tail -3
!cp {SRC}/paper/article.pdf {SHARED}/article.pdf && echo saved {SHARED}/article.pdf